# Public historical notebook
Outputs, authentication metadata and personal artifact links have been removed for publication.
This is a historical code record, NOT a complete retraining kit. Do not Run All.
See the stage README and aggregate training history. Private datasets and model archives are not included.


# Colab arxiv nusxasi

Bu VS Code’da ko‘rsatish uchun saqlangan tarixiy notebook. Bu ishga tushirish shabloni; bajarilgan tarix deb ko‘rsatilmaydi. Maxfiy tokenlar yashiriladi, HTML/rasm chiqishlari olinmaydi. Treningni laptopda Run All qilmang. Haqiqiy yangi trening uchun Colab va alohida run kerak.


# 3551 modeldan 10 000 juftlik bilan 1 epoch

**Alohida notebook. L4 GPU. Eski treninglar barcha sessiyalarda to‘xtagan bo‘lsin.**
Eski run/data/backup o‘zgarmaydi. Yangi optimizer/scheduler, taxminan 313 qadam.
Bu data avtomatik filtrlangan, to‘liq inson tekshirgan Gold emas.
Colab GPU compute unit sarflanadi; Gemini API ishlatilmaydi.
Save failed notebook xatosi bilan model backupini aralashtirmang.

## 1. ZIPni yuklash va tekshirish

Katakni bajaring va berilgan `uznorm-train-10k-v1.zip` faylni tanlang.
Google Drive mount kerak emas. Asl trening ZIP/notebookini almashtirmang.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, stat, subprocess, sys, tempfile, zipfile
from google.colab import files

ZIP = Path('/content/uznorm-train-10k-v1.zip')
EXPECTED_SHA256 = '51a76be0889ccbe7f5eb9d762304ab3b9decef661787aaf5a86653e220b31b49'
if not ZIP.is_file():
    files.upload()
if not ZIP.is_file():
    raise RuntimeError('uznorm-train-10k-v1.zip faylini tanlang.')
if hashlib.sha256(ZIP.read_bytes()).hexdigest() != EXPECTED_SHA256:
    raise RuntimeError('ZIP SHA-256 mos emas; boshqa paketni ishlatmang.')
KIT = Path(tempfile.mkdtemp(prefix='uznorm-10k-kit-', dir='/content'))
with zipfile.ZipFile(ZIP) as z:
    infos = z.infolist()
    if len(infos) > 200 or len(set(i.filename for i in infos)) != len(infos) or sum(i.file_size for i in infos) > 100*1024**2:
        raise RuntimeError('Noto‘g‘ri paket tarkibi/hajmi.')
    for info in infos:
        name = info.filename
        p = Path(name)
        if p.is_absolute() or '..' in p.parts or ':' in name or chr(92) in name or info.is_dir() or stat.S_ISLNK(info.external_attr >> 16):
            raise RuntimeError('Xavfli ZIP yo‘li.')
        target = KIT / p
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as src, target.open('xb') as dst:
            shutil.copyfileobj(src, dst)
manifest = json.loads((KIT / 'PACKAGE.json').read_text())
for name, expected in manifest['files'].items():
    if hashlib.sha256((KIT / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError('Paket a’zosi hash xatosi: ' + name)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(KIT/'requirements-colab.txt')], check=True)
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(KIT) + os.pathsep + str(KIT/'src')
ENV['USE_TF'] = '0'
ENV['USE_FLAX'] = '0'
subprocess.run([sys.executable, '-u', '-m', 'stage10k.runner', '--package', str(KIT), '--check-only'], env=ENV, check=True)
print('PAKET TAYYOR:', KIT)


## 2. Google hisob va W&B

Chapdagi 🔑 **Secrets**: `WANDB_API_KEY` → shu notebook uchun **Notebook access**.
Keyni matn sifatida kodga yozmang. Katakni bajaring va eski backup turgan
Google hisobga ruxsat bering. Bu bosqich treningni hali boshlamaydi.


In [ ]:
from google.colab import auth, userdata
auth.authenticate_user()
# Authenticate may update credential environment variables; pass the fresh environment.
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(KIT) + os.pathsep + str(KIT/'src')
ENV['USE_TF'] = '0'
ENV['USE_FLAX'] = '0'
try:
    key = userdata.get('WANDB_API_KEY').strip()
except Exception:
    raise RuntimeError('Secrets: WANDB_API_KEY va shu notebook uchun Notebook access kerak.') from None
if not key:
    raise RuntimeError('WANDB_API_KEY bo‘sh.')
ENV['WANDB_API_KEY'] = key
del key
print('Ruxsatlar tayyor. Kalit ekranga chiqarilmadi.')


## 3. Treningni boshlash / shu bosqichni davom ettirish

`START_NEW_STAGE = True` qiling va **shu katakni bajaring**.
Bu yangi kodni ishga tushirish, 10k non-Gold tanlovdan foydalanish, Colab GPU
sarfi hamda Drive’da alohida backup papka yaratishga tasdiqdir. Eski treninglar
to‘xtaganini yana tekshiring; ikki sessiyada parallel run qilmang.

Avval 3551 cloud backup (~4.34 GiB) yuklanib tekshiriladi, GPU sinovi va 128
misolli boshlang‘ich baholash bajariladi. Shundan so‘ng 1 epoch trening.
Har 50 qadamda model/optimizer/RNG to‘liq saqlanadi. Eski best model arxivga
qo‘shilmaydi. Taxminan 20 GiB yangi backup; avtomatik o‘chirish yo‘q.
Colab uzilsa, aynan shu paket bilan qayta bajaring: script **yangi stage**ning
oxirgi cloud checkpointidan qaytadi, eski 3551ga jim qaytmaydi.


In [ ]:
START_NEW_STAGE = False
if START_NEW_STAGE:
    command = [sys.executable, '-u', '-m', 'stage10k.runner', '--package', str(KIT),
               '--allow-cloud', '--old-stopped', '--allow-non-gold']
    process = subprocess.Popen(command, env=ENV)
    try:
        exit_code = process.wait()
    except BaseException:
        if process.poll() is None:
            import signal
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                print('Jarayon hali to‘xtamadi. Boshqa run ochmang; GPU holatini tekshiring.')
        raise
    print('Trening exit code:', exit_code)
    if exit_code:
        raise RuntimeError('Trening tugamadi. Yuqoridagi asl logni yuboring; eski checkpointlar o‘zgarmadi.')
else:
    print('Boshlash uchun START_NEW_STAGE = True qiling va shu katakni qayta bajaring.')


## Natija

`STAGE_COMPLETE_CLOUD_VERIFIED step=313` tugaganini bildiradi. Yakuniy arxivdagi
`run/comparison.json` oldin/keyin ko‘rsatkichlar, `run/after-predictions.jsonl`
xom javoblar. `checkpoint/` model/tokenizer bilan interaktiv sinov uchun ham,
optimizer/scheduler/RNG bilan **shu stage**ni tiklash uchun ham mos.
Natija yaxshilanganini tekshirmasdan yangi modelni eskisi o‘rniga qo‘ymang.
Ma’no saqlanishi inson tekshiruvini talab qiladi.
